# Sistemas Intensisvos en Datos - 202620

## Proyecto del Curso - 1ra Entrega

In [0]:
!pip install pathlib

## Índice
- [1. Contexto](#0-importación-de-librerías)
- [2. Ingesta de datos](#2-ingesta-de-datos)
- [3. Preguntas de negocio](#3-preguntas-de-negocio)

### 0. Importación de librerías

Se hace la importación de las librerias necesarias para la ejecución del _notebook_ relacionado con el proyecto de **OceanWatch Analytics**.

In [0]:
from pathlib import Path
import requests
import time
import zipfile


### 1. Ingesta de datos

In [0]:
# --- Configuración ---

# Creación del volumen
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.proyecto")

# URL para la descarga de los archivos en NOAA
base_url = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/"

# Archivos a descargar
files = [f"AIS_2023_06_{day:02d}.zip" for day in range(1, 8)]  # 01 a 07

# Directorios

# Directorio del volumen principal en Databricks
base_path = Path("/Volumes/workspace/default/proyecto/")

# Directorio para los archivos descargados
raw_dir = base_path / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Directorio para los archivos extraídos
dest = base_path / "extraido"
dest.mkdir(exist_ok=True)

# Configuración de descarga
MAX_RETRIES = 4
TIMEOUT = 60  # segundos por intento
CHUNK_SIZE = 1024 * 1024  # 1 MB


Definición de constantes

In [0]:
def verify_zip_integrity(path):
    """Verifica que el zip no esté corrupto revisando el CRC de cada archivo interno."""
    try:
        with zipfile.ZipFile(path) as z:
            bad_file = z.testzip()  # None si todo está bien; si no, devuelve el nombre del primer archivo corrupto
            if bad_file is not None:
                raise IOError(f"Archivo corrupto dentro del zip: {bad_file}")
            # Chequeo extra: que efectivamente tenga contenido (al menos 1 CSV)
            names = z.namelist()
            if not names:
                raise IOError("El zip está vacío")
            print(f"✓ Integridad OK: {path.name} ({len(names)} archivo(s) interno(s), CRC válido)")
            return True
    except zipfile.BadZipFile:
        raise IOError(f"{path.name} no es un zip válido (posiblemente descarga incompleta o corrupta)")

In [0]:

def download_with_retries(filename, max_retries=MAX_RETRIES):
    url = base_url + filename
    dest = raw_dir / filename

    # Si ya existe y no está vacío/corrupto, lo saltamos (idempotencia)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ {filename} ya existe ({dest.stat().st_size / 1e6:.1f} MB), se omite descarga")
        return dest

    for attempt in range(1, max_retries + 1):
        try:
            print(f"↓ Descargando {filename} (intento {attempt}/{max_retries})...")
            with requests.get(url, stream=True, timeout=TIMEOUT) as r:
                r.raise_for_status()
                expected_size = int(r.headers.get("Content-Length", 0))

                tmp_dest = dest.with_suffix(".part")
                with open(tmp_dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)

                actual_size = tmp_dest.stat().st_size

                # --- Verificación de integridad: tamaño esperado vs recibido ---
                if expected_size and actual_size != expected_size:
                    raise IOError(
                        f"Tamaño incompleto: esperado {expected_size} bytes, "
                        f"recibido {actual_size} bytes"
                    )

                tmp_dest.rename(dest)

                # --- Verificación de integridad (nivel 2: CRC interno del zip) ---
                verify_zip_integrity(dest)

                print(f"✓ {filename} descargado correctamente ({actual_size / 1e6:.1f} MB)")
                return dest

        except (requests.RequestException, IOError) as e:
            print(f"  ✗ Falló intento {attempt}: {e}")
            if tmp_dest.exists():
                tmp_dest.unlink()  # limpiar descarga parcial
            if attempt < max_retries:
                wait = 2 ** attempt  # backoff exponencial: 2, 4, 8, 16s
                print(f"  Reintentando en {wait}s...")
                time.sleep(wait)
            else:
                raise RuntimeError(f"No se pudo descargar {filename} tras {max_retries} intentos")

# --- Ejecutar descarga de los 7 días ---
downloaded = []
for f in files:
    path = download_with_retries(f)
    downloaded.append(path)

print(f"\nTotal descargado: {len(downloaded)} archivos")

### Extracción de los archivos

In [0]:
for path in raw_dir.glob("*.zip"):
    target_dir = dest / path.stem

    # Si ya existe y tiene archivos adentro, asumimos que ya se descomprimió
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"↷ {path.name} ya está descomprimido, se omite ({target_dir.name})")
        continue

    target_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(target_dir)
    print(f"✓ extraído: {path.name}")

![image_1789760447180.png](./image_1789760447180.png "image_1789760447180.png")

In [0]:
schema = (
    "MMSI string, "
    "BaseDateTime timestamp, "
    "LAT double, "
    "LON double, "
    "SOG float, "
    "COG float, "
    "Heading float, "
    "VesselName string, "
    "IMO string, "
    "CallSign string, "
    "VesselType smallint, "
    "Status smallint, "
    "Length float, "
    "Width float, "
    "Draft float, "
    "Cargo string, "
    "TransceiverClass string"
)

In [0]:
data_points = spark.read.csv(
    str(dest / "*" / "*.csv"),
    header=True,
    schema=schema,
    sep=",",
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True,
    dateFormat="yyyy-MM-dd'T'HH:mm:ss",
)


In [0]:
display(data_points)

In [0]:
from pyspark.sql import functions as f

# Número de particiones, sin tocar RDD
data_points.select(f.spark_partition_id().alias("part_id")).distinct().count()

In [0]:
# Filas por partición (la que ya usamos para detectar skew) — esta sí funciona en serverless,
# porque spark_partition_id() es una función de columna normal, no toca la API de RDD
data_points.withColumn("part_id", f.spark_partition_id()) \
    .groupBy("part_id").count().orderBy("part_id").show(50)


### 2. Exploración & Perfilamiento